In [1]:
%pip install sqlglot

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import csv
import pandas as pd
import sqlglot
from sqlglot import exp

In [3]:
INPUT_FILE = '../asset/result.csv'
CLEANED_FILE = '../asset/cleaned_result.csv'

In [4]:
with open(INPUT_FILE, 'r', encoding='utf-8') as f_in, \
     open(CLEANED_FILE, 'w', encoding='utf-8', newline='') as f_out:
    
    # Reader này sẽ tự động xử lý các dòng có nội dung nằm trong ngoặc kép
    reader = csv.reader(f_in, quotechar='"', delimiter=',', 
                        skipinitialspace=True, quoting=csv.QUOTE_MINIMAL)
    writer = csv.writer(f_out)
    
    for row in reader:
        # Loại bỏ các ký tự xuống dòng dư thừa bên trong nội dung SQL để làm sạch
        clean_row = [col.replace('\n', ' ').replace('\r', ' ').strip() if isinstance(col, str) else col for col in row]
        writer.writerow(clean_row)

In [5]:
input = pd.read_csv(CLEANED_FILE,
    on_bad_lines='warn', 
    lineterminator='\n',
    dtype=str, 
    low_memory=False)

In [6]:
print(len(input))

1063232


In [7]:
print(input['statement'].unique()[1:10])
unique_statments = list(input['statement'].unique())
print(f"Total unique statements: {len(unique_statments)}")

<ArrowStringArray>
[  'SELECT TOP 6 p.ra AS pra, p.dec AS pdec, p.type, p.u, p.g, p.r, p.i, p.z, p.psfMag_u, p.psfMag_g, p.psfMag_r, p.psfMag_i, p.psfMag_z, s.z AS redshift, s.class AS spec_class, s.zWarning FROM PhotoObjAll AS p JOIN SpecObjAll AS s ON p.objID = s.bestObjID WHERE p.ra BETWEEN 186.92024673201263 AND 186.92108006534596 AND p.dec BETWEEN 33.36578861889853 AND 33.36662195223186 AND s.zWarning = 0',
    'SELECT TOP 6 p.ra AS pra, p.dec AS pdec, p.type, p.u, p.g, p.r, p.i, p.z, p.psfMag_u, p.psfMag_g, p.psfMag_r, p.psfMag_i, p.psfMag_z, s.z AS redshift, s.class AS spec_class, s.zWarning FROM PhotoObjAll AS p JOIN SpecObjAll AS s ON p.objID = s.bestObjID WHERE p.ra BETWEEN 186.7662171533437 AND 186.76705048667702 AND p.dec BETWEEN 33.17735659218348 AND 33.17818992551681 AND s.zWarning = 0',
 'SELECT TOP 6 p.ra AS pra, p.dec AS pdec, p.type, p.u, p.g, p.r, p.i, p.z, p.psfMag_u, p.psfMag_g, p.psfMag_r, p.psfMag_i, p.psfMag_z, s.z AS redshift, s.class AS spec_class, s.zWarning 

In [8]:
# check for syntax errors and delete by sqlglot
def is_valid_sql(statement):
    if not isinstance(statement, str):
        return False
    
    try:
        sqlglot.parse_one(statement, read="tsql")
        return True
    except Exception:
        return False

In [9]:
for statement in unique_statments:
    if not is_valid_sql(statement):
        unique_statments.remove(statement)

'SET PARSEONLY ON select b.objID,b.psfMag_u, b.psfMag_g, b.psfMag_r, b.psfMag_i, b.psfMag_z, b.extinc' contains unsupported syntax. Falling back to parsing as a 'Command'.
'CREATE TABLE #upload ( up_id int, up_ra float, up_dec float )  INSERT INTO #upload values ( 1, 132.8' contains unsupported syntax. Falling back to parsing as a 'Command'.
'CREATE TABLE #upload ( up_id int, up_ra float, up_dec float )  INSERT INTO #upload values ( 1, 337.5' contains unsupported syntax. Falling back to parsing as a 'Command'.
'CREATE TABLE #upload ( up_id int, up_ra float, up_dec float )  INSERT INTO #upload values ( 1, 147.4' contains unsupported syntax. Falling back to parsing as a 'Command'.
'CREATE TABLE #upload ( up_id int, up_ra float, up_dec float )  INSERT INTO #upload values ( 1, 161.3' contains unsupported syntax. Falling back to parsing as a 'Command'.
'CREATE TABLE #upload ( up_id int, up_ra float, up_dec float )  INSERT INTO #upload values ( 1, 187.2' contains unsupported syntax. Falling 

In [10]:
#Get 200 statements only 
unique_statments = unique_statments[:200]

In [11]:
ast_statements = []

In [12]:
print(unique_statments[0])

SELECT TOP 6 p.ra AS pra, p.dec AS pdec, p.type, p.u, p.g, p.r, p.i, p.z, p.psfMag_u, p.psfMag_g, p.psfMag_r, p.psfMag_i, p.psfMag_z, s.z AS redshift, s.class AS spec_class, s.zWarning FROM PhotoObjAll AS p JOIN SpecObjAll AS s ON p.objID = s.bestObjID WHERE p.ra BETWEEN 186.43365814387732 AND 186.43449147721066 AND p.dec BETWEEN 33.738606352550406 AND 33.73943968588374 AND s.zWarning = 0


In [13]:
print(sqlglot.parse_one(unique_statments[0], read="tsql").dump())

[{'c': 'sqlglot.expressions.query.Select'}, {'i': 0, 'k': 'expressions', 'a': True, 'c': 'sqlglot.expressions.core.Alias'}, {'i': 1, 'k': 'this', 'c': 'sqlglot.expressions.core.Column'}, {'i': 2, 'k': 'this', 'c': 'sqlglot.expressions.core.Identifier', 'm': {'line': 1, 'col': 17, 'start': 15, 'end': 16}}, {'i': 3, 'k': 'this', 'v': 'ra'}, {'i': 3, 'k': 'quoted', 'v': False}, {'i': 2, 'k': 'table', 'c': 'sqlglot.expressions.core.Identifier', 'm': {'line': 1, 'col': 14, 'start': 13, 'end': 13}}, {'i': 6, 'k': 'this', 'v': 'p'}, {'i': 6, 'k': 'quoted', 'v': False}, {'i': 1, 'k': 'alias', 'c': 'sqlglot.expressions.core.Identifier', 'm': {'line': 1, 'col': 24, 'start': 21, 'end': 23}}, {'i': 9, 'k': 'this', 'v': 'pra'}, {'i': 9, 'k': 'quoted', 'v': False}, {'i': 0, 'k': 'expressions', 'a': True, 'c': 'sqlglot.expressions.core.Alias'}, {'i': 12, 'k': 'this', 'c': 'sqlglot.expressions.core.Column'}, {'i': 13, 'k': 'this', 'c': 'sqlglot.expressions.core.Identifier', 'm': {'line': 1, 'col': 31,

In [14]:
index = 0
for statement in unique_statments:
    print(f"Processing statement {index + 1}/{len(unique_statments)}")
    print(f"Statement: {statement}")
    ast = sqlglot.parse_one(statement, read="tsql")

    features = {
        "tables": [t.name.lower() for t in ast.find_all(exp.Table)],
        
        # TRÍCH XUẤT CỘT TRONG JOIN ON
        "join_columns": [
                c.name.lower() 
                for j in ast.find_all(exp.Join) 
                if j.args.get("on") 
                for c in j.args["on"].find_all(exp.Column)
        ],

        "columns_where": [c.name.lower() for c in ast.find(exp.Where).find_all(exp.Column)] if ast.find(exp.Where) else [],
        "columns_group": [c.name.lower() for c in ast.find(exp.Group).find_all(exp.Column)] if ast.find(exp.Group) else [],
        "join_types": [j.args.get("side") for j in ast.find_all(exp.Join) if j.args.get("side")],
        "timestamp": index
   }

    index += 1

    ast_statements.append(features)

Processing statement 1/200
Statement: SELECT TOP 6 p.ra AS pra, p.dec AS pdec, p.type, p.u, p.g, p.r, p.i, p.z, p.psfMag_u, p.psfMag_g, p.psfMag_r, p.psfMag_i, p.psfMag_z, s.z AS redshift, s.class AS spec_class, s.zWarning FROM PhotoObjAll AS p JOIN SpecObjAll AS s ON p.objID = s.bestObjID WHERE p.ra BETWEEN 186.43365814387732 AND 186.43449147721066 AND p.dec BETWEEN 33.738606352550406 AND 33.73943968588374 AND s.zWarning = 0
Processing statement 2/200
Statement: SELECT TOP 6 p.ra AS pra, p.dec AS pdec, p.type, p.u, p.g, p.r, p.i, p.z, p.psfMag_u, p.psfMag_g, p.psfMag_r, p.psfMag_i, p.psfMag_z, s.z AS redshift, s.class AS spec_class, s.zWarning FROM PhotoObjAll AS p JOIN SpecObjAll AS s ON p.objID = s.bestObjID WHERE p.ra BETWEEN 186.92024673201263 AND 186.92108006534596 AND p.dec BETWEEN 33.36578861889853 AND 33.36662195223186 AND s.zWarning = 0
Processing statement 3/200
Statement: SELECT TOP 6 p.ra AS pra, p.dec AS pdec, p.type, p.u, p.g, p.r, p.i, p.z, p.psfMag_u, p.psfMag_g, p.psf

In [15]:
print(ast_statements[:5])

[{'tables': ['photoobjall', 'specobjall'], 'join_columns': ['objid', 'bestobjid'], 'columns_where': ['zwarning', 'ra', 'dec'], 'columns_group': [], 'join_types': [], 'timestamp': 0}, {'tables': ['photoobjall', 'specobjall'], 'join_columns': ['objid', 'bestobjid'], 'columns_where': ['zwarning', 'ra', 'dec'], 'columns_group': [], 'join_types': [], 'timestamp': 1}, {'tables': ['photoobjall', 'specobjall'], 'join_columns': ['objid', 'bestobjid'], 'columns_where': ['zwarning', 'ra', 'dec'], 'columns_group': [], 'join_types': [], 'timestamp': 2}, {'tables': ['photoobjall', 'specobjall'], 'join_columns': ['objid', 'bestobjid'], 'columns_where': ['zwarning', 'ra', 'dec'], 'columns_group': [], 'join_types': [], 'timestamp': 3}, {'tables': ['photoobjall', 'specobjall'], 'join_columns': ['objid', 'bestobjid'], 'columns_where': ['zwarning', 'ra', 'dec'], 'columns_group': [], 'join_types': [], 'timestamp': 4}]


In [16]:
import json

with open('ast_statements.json', 'w') as f:
    json.dump(ast_statements, f, indent=4)
    